In [2]:
import numpy as np
from optimal_comparison_plots import solve_once

# Example usage of solve_once
# Define example TSA parameters (you should adapt these for your test case)
voltage_base = 6
params = {
    'stroke': 0.100,            # m
    'm': 0.080,                  # kg
    'I': 4.5e-6,               # kg·m^2
    'tau_max': 0.09555555556 * voltage_base,    # N·m (example, adjust as needed)
    'w_max': 2135*2*np.pi/60 * voltage_base,            # rad/s (example, adjust as needed)
    'g': 9.81,                 # m/s^2
    'L_min': 0.144,
    'L_max': 0.18,
}
guess = [0.12, 0.004, 0.03]   # [L, r, T]
N = 40                        # Number of trajectory nodes
max_iter = 1000
tol = 1e-6
theta_margin = 0.98

# Run a single optimization
result = solve_once(params, guess, N, max_iter, tol, theta_margin)
print("Optimization Result:")
for k, v in result.items():
    print(f"  {k}: {v}")


Optimization Result:
  success: True
  L: 0.1439999900125465
  r: 0.006690381879202948
  T: 0.028480572381821025
  y_final: 0.09999999999999881
  ydot_final: 10.111197982307127


In [56]:
from plot_optimal_cases import integrate_theta

# Use the optimize result to perform numerical integration
# Build get_y and get_tau functions using the result (which are L, r, T)

# L = result['L']
# r = result['r']

L = 0.100
r = 0.008

voltage_base = 7.3
params = {
    'stroke': 0.050,            # m
    'm': 0.075,                  # kg
    'I': 4.5e-6,               # kg·m^2
    'tau_max': 0.09555555556 * voltage_base,    # N·m (example, adjust as needed)
    'w_max': 2135*2*np.pi/60 * voltage_base,            # rad/s (example, adjust as needed)
    'g': 9.81,                 # m/s^2
}

def get_y(theta):
    # Returns (y, dy/dtheta, d2y/dtheta2)
    L2 = L**2
    rt = r * theta
    inside = L2 - rt**2
    sqrt_inside = np.sqrt(max(inside, 1e-12))
    y = L - sqrt_inside
    dy_dtheta = (r**2 * theta) / sqrt_inside
    d2y_dtheta2 = (r**2 * L2) / max(inside, 1e-18)**1.5
    return y, dy_dtheta, d2y_dtheta2

def get_tau(theta, theta_dot, t):
    # From TSA motor model - torque limited and speed limited
    tau_max = params['tau_max']
    w_max = params['w_max']
    return tau_max * (1 - theta_dot / w_max)

# Integrate the trajectory using the optimized values
sim_result = integrate_theta(get_y, get_tau, params)

body_vel = sim_result['y_d'][-1]
print("Simulated Trajectory:")
print(f"  Final y: {sim_result['y'][-1]:.5f} m")
print(f"  Final y_dot: {body_vel:.5f} m/s")

#inelastic collision with a mass
m_foot = 0.020
v_takeoff = params['m'] * body_vel / (params['m'] + m_foot)
print(f"  Takeoff velocity: {v_takeoff:.5f} m/s")

#calculate the height of the takeoff
h_takeoff = v_takeoff**2 / (2 * params['g'])
print(f"  Height: {h_takeoff:.5f} m")

Simulated Trajectory:
  Final y: 0.04977 m
  Final y_dot: 9.00474 m/s
  Takeoff velocity: 7.10900 m/s
  Height: 2.57584 m
